# SkillMap Stage 2: skill relationship model

Trains a cross-encoder that reads two skill names (A = the student's skill, B = the job's skill) and predicts
**SATISFIES** (SAME + NARROWER), **RELATED** (RELATED + BROADER) or **NOT** (DIFFERENT), following
`docs/skill_relation_label_guide.md`. BROADER moved from NOT to RELATED on 2 Oct (references.md "Where BROADER belongs");
the 3 classes are always rebuilt here from the 5-way `label5`, so older CSVs follow the same rule.

**Which training set** (set `PAIRS_CSV` in cell 1, one file or several joined with a comma):
- `pairs_esco_onet_v1.csv` = v1 (ESCO / O*NET wording, first run)
- `pairs_skillmap_v2.csv` = v2 only (SkillMap's own skill names, labelled by gpt-oss + Qwen)
- `pairs_esco_onet_v1.csv,pairs_skillmap_v2.csv` = v1 + v2 together
- `pairs_esco_onet_v1.csv,pairs_skillmap_v2.csv,pairs_skillmap_v3.csv` = run 5: adds the targeted v3 pairs (extend + sibling, 6 Oct). `V2_REPEAT` repeats every SkillMap row (v2 and v3)

The test split printed here is only a first check. The real test is the 400 reference pairs, run on your laptop with
`evaluate_reference.py --model <folder>` (never used for training).

Run on Google Colab: *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. Upload the CSV(s) when asked.
Each seed trains for a few minutes on a T4.

In [ ]:
# 1. Setup
!pip -q install transformers scikit-learn pandas
import os, json, random, time
import numpy as np, pandas as pd, torch
from torch import nn
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, classification_report, confusion_matrix

SBERT_NAME = "sentence-transformers/all-MiniLM-L6-v2"                 # the app's SBERT model (cosine baseline)
# Model to fine-tune: the SBERT model (as before) or a cross-encoder already trained on sentence pairs (NLI:
# "does A imply B?"), which starts out knowing that look-alike wording is not the same meaning
MODEL_NAME = os.environ.get("MODEL_NAME", SBERT_NAME)   # or "cross-encoder/nli-MiniLM2-L6-H768"
# Re-weighting (3 Oct): in v1 only 4% of NOT pairs share a word (73% of SATISFIES do), so the model learned
# "shared word = has it". V2_REPEAT repeats the SkillMap pairs in training; V1_EASY_NOT_KEEP keeps that share of
# v1's NOT pairs with no shared word. 1 and 1.0 = the original runs
V2_REPEAT = int(os.environ.get("V2_REPEAT", 1))
V1_EASY_NOT_KEEP = float(os.environ.get("V1_EASY_NOT_KEEP", 1.0))
PAIRS_CSV = os.environ.get("PAIRS_CSV", "pairs_skillmap_v2.csv")     # see the note above: v2 only, or "v1,v2" files
RUN_NAME = os.environ.get("RUN_NAME", "")                            # name for the saved model; empty = from the files
SEEDS = [int(s) for s in os.environ.get("SEEDS", "13,42,77").split(",")]
EPOCHS, LR, BATCH, MAX_LEN = int(os.environ.get("EPOCHS", 10)), 2e-5, 32, 64
LABELS = ["SATISFIES", "RELATED", "NOT"]
TO_CLASS = {"SAME": "SATISFIES", "NARROWER": "SATISFIES", "RELATED": "RELATED", "BROADER": "RELATED",
            "DIFFERENT": "NOT"}                                      # same mapping as merge_*_labels.py
FILES = [f.strip() for f in PAIRS_CSV.split(",") if f.strip()]
RUN_NAME = RUN_NAME or "_".join(os.path.splitext(os.path.basename(f))[0].replace("pairs_", "") for f in FILES)
if MODEL_NAME != SBERT_NAME and "nli" not in RUN_NAME:
    RUN_NAME += "_nli" if "nli" in MODEL_NAME else "_" + MODEL_NAME.split("/")[-1]
if (V2_REPEAT, V1_EASY_NOT_KEEP) != (1, 1.0) and "_rw" not in RUN_NAME:
    RUN_NAME += f"_rw{V2_REPEAT}x{int(V1_EASY_NOT_KEEP * 100)}"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| model:", MODEL_NAME, "| v2 x", V2_REPEAT, "| v1 easy NOT kept", V1_EASY_NOT_KEEP, "| training files:", FILES, "| run:", RUN_NAME)

In [ ]:
# 2. Data (upload the CSV(s) in Colab if they aren't there yet)
missing = [f for f in FILES if not os.path.exists(f)]
if missing:
    from google.colab import files
    print("Upload:", missing)
    files.upload()
    still = [f for f in FILES if not os.path.exists(f)]
    assert not still, f"still missing {still}: upload files with exactly these names"
df = pd.concat([pd.read_csv(f).assign(file=os.path.basename(f)) for f in FILES], ignore_index=True)
df["label3"] = df.label5.map(TO_CLASS)                  # rebuilt from label5: v1's old label3 had BROADER as NOT
assert df.label3.notna().all(), "unknown label5 values: " + str(set(df.label5) - set(TO_CLASS))
df["y"] = df.label3.map({l: i for i, l in enumerate(LABELS)})
train, val, test = (df[df.split == s].reset_index(drop=True) for s in ("train", "val", "test"))
print(pd.crosstab(df.label5, df.split, margins=True))
print(pd.crosstab(df.file, df.split, margins=True))

# Re-weighting, TRAIN split only (val and test stay as they are, so scores stay comparable)
import re
STOP = {"and", "of", "the", "for", "in", "on", "to", "a", "an", "with", "skills", "skill", "management",
        "development", "system", "systems", "design", "engineering", "knowledge", "experience", "&"}
words = lambda s: {w for w in re.findall(r"[a-z0-9+#]+", str(s).lower()) if w not in STOP and len(w) > 1}
train["shares_word"] = [bool(words(a) & words(b)) for a, b in zip(train.a, train.b)]
def not_share(frame):
    n = frame[frame.label3 == "NOT"]
    return f"{n.shares_word.mean():.0%} of {len(n)} NOT pairs share a word"
print("train before:", not_share(train))
is_v1 = train.file.str.contains("esco_onet")
easy = is_v1 & (train.label3 == "NOT") & ~train.shares_word
drop = train[easy].sample(frac=1 - V1_EASY_NOT_KEEP, random_state=0).index
train = train.drop(drop)
train = pd.concat([train] + [train[~train.file.str.contains("esco_onet")]] * (V2_REPEAT - 1), ignore_index=True)
print("train after: ", not_share(train), f"| {len(drop)} easy v1 NOT pairs dropped, v2 rows x{V2_REPEAT}, {len(train)} rows")
print(pd.crosstab(train.file, train.label3, margins=True))

In [ ]:
# 3. Baseline: what the app does today (SBERT cosine, 0.7 = has the skill, 0.6-0.7 = related)
sbert_tok = AutoTokenizer.from_pretrained(SBERT_NAME)
enc = AutoModel.from_pretrained(SBERT_NAME).to(DEVICE).eval()

@torch.no_grad()
def embed(texts, bs=128):
    out = []
    for i in range(0, len(texts), bs):
        b = sbert_tok(texts[i:i + bs], padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        h = enc(**b).last_hidden_state
        m = b["attention_mask"].unsqueeze(-1).float()
        v = (h * m).sum(1) / m.sum(1)                      # mean pooling, as sentence-transformers does
        out.append(torch.nn.functional.normalize(v, dim=-1).cpu())
    return torch.cat(out).numpy()

def cosine_predict(frame, hi=0.7, lo=0.6):
    sims = (embed(frame.a.tolist()) * embed(frame.b.tolist())).sum(1)
    return np.where(sims >= hi, 0, np.where(sims >= lo, 1, 2)), sims

base_pred, base_sims = cosine_predict(test)
baseline = {"macro_f1": f1_score(test.y, base_pred, average="macro")}
print("Baseline (cosine 0.7 / 0.6) macro-F1 on test:", round(baseline["macro_f1"], 3))
print(classification_report(test.y, base_pred, target_names=LABELS, zero_division=0))
del enc

In [ ]:
# 4. Train the cross-encoder (3 seeds)
tok = AutoTokenizer.from_pretrained(MODEL_NAME)        # the fine-tuned model's own tokeniser
def batches(frame, shuffle, seed=0):
    idx = list(range(len(frame)))
    if shuffle:
        random.Random(seed).shuffle(idx)
    for i in range(0, len(idx), BATCH):
        part = frame.iloc[idx[i:i + BATCH]]
        b = tok(part.a.tolist(), part.b.tolist(), padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
        yield {k: v.to(DEVICE) for k, v in b.items()}, torch.tensor(part.y.values, device=DEVICE)

@torch.no_grad()
def predict(model, frame):
    model.eval()
    preds = []
    for b, _ in batches(frame, False):
        preds.append(model(**b).logits.argmax(-1).cpu())
    return torch.cat(preds).numpy() if preds else np.array([])

counts = np.bincount(train.y, minlength=3)
weights = torch.tensor(len(train) / (3 * np.maximum(counts, 1)), dtype=torch.float, device=DEVICE)  # class weights
runs = []
for seed in SEEDS:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3, id2label=dict(enumerate(LABELS)), label2id={l: i for i, l in enumerate(LABELS)},
        ignore_mismatched_sizes=True).to(DEVICE)   # an NLI model's own 3 labels are replaced by ours
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss(weight=weights)
    best, best_state, t0 = -1, None, time.time()
    for epoch in range(EPOCHS):
        model.train()
        for b, y in batches(train, True, seed + epoch):
            loss = loss_fn(model(**b).logits, y)
            loss.backward(); opt.step(); opt.zero_grad()
        f1 = f1_score(val.y, predict(model, val), average="macro")
        print(f"seed {seed} epoch {epoch + 1}: val macro-F1 {f1:.3f}")
        if f1 > best:
            best, best_state = f1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    pred = predict(model, test)
    runs.append({"seed": seed, "val_f1": best, "test_f1": f1_score(test.y, pred, average="macro"),
                 "pred": pred.tolist(), "minutes": round((time.time() - t0) / 60, 1)})
    if seed == SEEDS[0]:
        model.save_pretrained(f"relation_model_{RUN_NAME}"); tok.save_pretrained(f"relation_model_{RUN_NAME}")
    print(f"seed {seed}: test macro-F1 {runs[-1]['test_f1']:.3f} ({runs[-1]['minutes']} min)")

In [ ]:
# 5. Results
f1s = [r["test_f1"] for r in runs]
print(f"Cross-encoder test macro-F1: {np.mean(f1s):.3f} ± {np.std(f1s):.3f} over {len(f1s)} seeds")
print(f"Baseline (cosine) test macro-F1: {baseline['macro_f1']:.3f}")
pred = np.array(runs[0]["pred"])
print(classification_report(test.y, pred, target_names=LABELS, zero_division=0))
print("Confusion matrix (rows = true, cols = predicted):", LABELS)
print(confusion_matrix(test.y, pred, labels=[0, 1, 2]))

# Direction test: NARROWER must come out SATISFIES and the same pair reversed (BROADER) must not
d = test[test.label5.isin(["NARROWER", "BROADER"])]
ok = ((d.label5 == "NARROWER") & (pred[d.index] == 0)) | ((d.label5 == "BROADER") & (pred[d.index] != 0))
cos_ok = ((d.label5 == "NARROWER") & (base_pred[d.index] == 0)) | ((d.label5 == "BROADER") & (base_pred[d.index] != 0))
print(f"Direction test ({len(d)} pairs): model {ok.mean():.3f}, cosine {cos_ok.mean():.3f} "
      "(cosine gives A->B and B->A the same score, so it can't exceed 0.5 on balanced pairs)")

json.dump({"run": RUN_NAME, "pairs_csv": FILES, "mapping": TO_CLASS, "model": MODEL_NAME, "seeds": SEEDS, "epochs": EPOCHS,
           "baseline_macro_f1": baseline["macro_f1"], "test_macro_f1": f1s,
           "direction_model": float(ok.mean()), "direction_cosine": float(cos_ok.mean()),
           "runs": [{k: v for k, v in r.items() if k != "pred"} for r in runs]},
          open(f"results_{RUN_NAME}.json", "w"), indent=1)
print(f"Saved results_{RUN_NAME}.json and relation_model_{RUN_NAME}/ (first seed)")

In [ ]:
# 6. Download the results (Colab)
try:
    from google.colab import files
    !zip -qr relation_model_{RUN_NAME}.zip relation_model_{RUN_NAME} results_{RUN_NAME}.json
    files.download(f"relation_model_{RUN_NAME}.zip")
except ImportError:
    print(f"Not on Colab: results_{RUN_NAME}.json and relation_model_{RUN_NAME}/ are in the current folder")